# 04 — Baseline modeling (run only after leakage review)

**Gate:** do not run this notebook until Notebook 02's leakage decisions and Notebook 03's feature provenance register are complete. This notebook provides a baseline, not a final research claim.

## 1. Load data and configure approved features

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np

# Put the CSV beside this notebook, or change DATA_PATH to its full path.
DATA_PATH = Path("metaverse_transactions_dataset.csv")

if not DATA_PATH.exists():
    candidates = list(Path(".").glob("*.csv"))
    raise FileNotFoundError(
        f"Could not find {DATA_PATH}. Place the original CSV beside this notebook "
        f"or edit DATA_PATH. CSV files in this folder: {[p.name for p in candidates]}"
    )

df = pd.read_csv(DATA_PATH)
print("Shape:", df.shape)
display(df.head())

Shape: (78600, 14)


,timestamp,hour_of_day,sending_address,receiving_address,amount,transaction_type,location_region,ip_prefix,login_frequency,session_duration,purchase_pattern,age_group,risk_score,anomaly
0,2022-04-11 12:47:27,12,0x9d32d0bf2c00f41ce7ca01b66e174cc4dcb0c1da,0x39f82e1c09bc6d7baccc1e79e5621ff812f50572,796.949206,transfer,Europe,192.000,3,48,focused,established,18.75,low_risk
1,2022-06-14 19:12:46,19,0xd6e251c23cbf52dbd472f079147873e655d8096f,0x51e8fbe24f124e0e30a614e14401b9bbfed5384c,0.010000,purchase,South America,172.000,5,61,focused,established,25.00,low_risk
2,2022-01-18 16:26:59,16,0x2e0925b922fed01f6a85d213ae2718f54b8ca305,0x52c7911879f783d590af45bda0c0ef2b8536706f,778.197390,purchase,Asia,192.168,3,74,focused,established,31.25,low_risk
3,2022-06-15 09:20:04,9,0x93efefc25fcaf31d7695f28018d7a11ece55457f,0x8ac3b7bd531b3a833032f07d4e47c7af6ea7bace,300.838358,transfer,South America,172.000,8,111,high_value,veteran,36.75,low_risk
4,2022-02-18 14:35:30,14,0xad3b8de45d63f5cce28aef9a82cf30c397c6ceb9,0x6fdc047c2391615b3facd79b4588c7e9106e49f2,775.569344,sale,Africa,172.160,6,100,high_value,veteran,62.50,moderate_risk


In [2]:
TARGET = "anomaly"

# Replace this list with ONLY the features approved in the provenance register.
# The default below excludes known/suspected label proxies and is deliberately conservative.
EXCLUDE = {
    TARGET, "risk_score", "transaction_type", "purchase_pattern", "age_group"
}
FEATURES = [c for c in df.columns if c not in EXCLUDE]

print("Features used:", FEATURES)
print("Target distribution:")
display(df[TARGET].value_counts())

Features used: ['timestamp', 'hour_of_day', 'sending_address', 'receiving_address', 'amount', 'location_region', 'ip_prefix', 'login_frequency', 'session_duration']
Target distribution:


,count
anomaly,
low_risk,63494
moderate_risk,8611
high_risk,6495


## 2. Chronological split

In [3]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler, LabelEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.metrics import classification_report, confusion_matrix, balanced_accuracy_score
from sklearn.ensemble import RandomForestClassifier

work = df[FEATURES + [TARGET]].copy()
if "timestamp" in work.columns:
    work["_sort_time"] = pd.to_datetime(work["timestamp"], errors="coerce", utc=True)
    work = work.sort_values("_sort_time").drop(columns="_sort_time")
    cut = int(len(work) * 0.8)
    train, test = work.iloc[:cut], work.iloc[cut:]
else:
    train, test = train_test_split(work, test_size=0.2, random_state=42, stratify=work[TARGET])

X_train, X_test = train[FEATURES], test[FEATURES]
y_train, y_test = train[TARGET].astype(str), test[TARGET].astype(str)

## 3. Leakage-safe preprocessing and baseline model

In [4]:
num_cols = X_train.select_dtypes(include=np.number).columns.tolist()
cat_cols = [c for c in FEATURES if c not in num_cols]

numeric_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])
categorical_pipe = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])
preprocess = ColumnTransformer([
    ("num", numeric_pipe, num_cols),
    ("cat", categorical_pipe, cat_cols)
])

model = Pipeline([
    ("preprocess", preprocess),
    ("classifier", RandomForestClassifier(
        n_estimators=300, class_weight="balanced",
        random_state=42, n_jobs=-1
    ))
])

model.fit(X_train, y_train)
pred = model.predict(X_test)

print("Balanced accuracy:", balanced_accuracy_score(y_test, pred))
print(classification_report(y_test, pred, zero_division=0))
print("Confusion matrix (label order):", model.classes_)
display(pd.DataFrame(
    confusion_matrix(y_test, pred, labels=model.classes_),
    index=model.classes_, columns=model.classes_
))

Balanced accuracy: 0.6360546241134406
               precision    recall  f1-score   support

    high_risk       0.29      0.07      0.12      1266
     low_risk       0.90      0.93      0.91     12742
moderate_risk       0.69      0.91      0.78      1712

     accuracy                           0.86     15720
    macro avg       0.63      0.64      0.60     15720
 weighted avg       0.83      0.86      0.83     15720

Confusion matrix (label order): ['high_risk' 'low_risk' 'moderate_risk']


,high_risk,low_risk,moderate_risk
high_risk,91,1175,0
low_risk,220,11817,705
moderate_risk,0,156,1556


## 4. Save evaluation outputs

In [5]:
from pathlib import Path
Path("outputs").mkdir(exist_ok=True)
pd.DataFrame(classification_report(y_test, pred, output_dict=True, zero_division=0)).T.to_csv(
    "outputs/04_classification_report.csv"
)
pd.DataFrame(
    confusion_matrix(y_test, pred, labels=model.classes_),
    index=model.classes_, columns=model.classes_
).to_csv("outputs/04_confusion_matrix.csv")
print("Saved outputs/04_classification_report.csv and outputs/04_confusion_matrix.csv")

Saved outputs/04_classification_report.csv and outputs/04_confusion_matrix.csv


## Interpretation requirements

- Report the split method and exact dates.
- Report class distribution in train and test.
- Compare against a majority-class baseline.
- Include per-class precision, recall, F1, confusion matrix, and PR-AUC where appropriate.
- Repeat with a justified alternative feature set only after provenance review.
- Do not claim deployment readiness from a single random-forest run.